# Brownian Motion and Multivariate Shifts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakeberv/bifrost/blob/main/vignettes/colab/theoretical-background-vignette.ipynb)

Converted from [`vignettes/theoretical-background-vignette.Rmd`](https://github.com/jakeberv/bifrost/blob/main/vignettes/theoretical-background-vignette.Rmd).

> **Recommended Colab runtime:** For compute-intensive examples, choose **Runtime > Change runtime type > v5e-1 TPU** when available. This notebook uses the runtime's host CPUs, not the TPU itself. Otherwise, choose the available runtime with the most CPUs.

In [ ]:
message("Detected logical CPUs: ", parallel::detectCores(logical = TRUE))
if (!dir.exists("/content/bifrost")) {
  system("git clone --depth 1 https://github.com/jakeberv/bifrost.git /content/bifrost")
}
colab_packages <- c("remotes", "knitr")
missing_packages <- colab_packages[
  !vapply(colab_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing_packages)) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}
remotes::install_local("/content/bifrost", dependencies = NA, upgrade = "never")
setwd("/content/bifrost/vignettes")


In [ ]:
knitr::opts_chunk$set(
  collapse = TRUE,
  comment = "#>",
  fig.align = "center",
  out.width = "100%",
  dpi = 170,
  fig.retina = 2,
  dev = "png"
)

as_trait_matrix <- function(sim) {
  out <- if (is.list(sim)) sim[[1]] else sim
  as.matrix(out)
}

descendant_tips <- function(tree, node) {
  children <- tree$edge[tree$edge[, 1] == node, 2]

  unlist(
    lapply(children, function(child) {
      if (child <= ape::Ntip(tree)) {
        child
      } else {
        descendant_tips(tree, child)
      }
    }),
    use.names = FALSE
  )
}

choose_shift_node <- function(tree, min_tips = 10, target_tips = 15) {
  internal_nodes <- sort(unique(tree$edge[, 1]))
  internal_nodes <- internal_nodes[internal_nodes > ape::Ntip(tree)]
  desc_counts <- vapply(
    internal_nodes,
    function(node) length(descendant_tips(tree, node)),
    integer(1)
  )

  eligible <- desc_counts >= min_tips
  if (!any(eligible)) {
    stop("No internal node meets the requested descendant-tip threshold.")
  }

  eligible_nodes <- internal_nodes[eligible]
  eligible_counts <- desc_counts[eligible]

  eligible_nodes[which.min(abs(eligible_counts - target_tips))]
}

draw_cov_ellipse <- function(center, sigma, level = 0.8, border = "black",
                             lwd = 2, lty = 1, lend = "round", n = 200) {
  ev <- eigen(sigma, symmetric = TRUE)
  theta <- seq(0, 2 * pi, length.out = n)
  circle <- rbind(cos(theta), sin(theta))
  radius <- sqrt(stats::qchisq(level, df = 2))
  shape <- ev$vectors %*% diag(sqrt(pmax(ev$values, 0)), nrow = 2)
  coords <- center + radius * shape %*% circle
  lines(coords[1, ], coords[2, ], col = border, lwd = lwd, lty = lty, lend = lend)
}

draw_pc_axes <- function(center, rotation, sdev, scale = 1.9,
                         cols = c("#252525", "#969696"), lwd = 2,
                         cex = 0.8, label_scale = 0.9,
                         show_labels = TRUE) {
  for (i in seq_len(ncol(rotation))) {
    vec <- rotation[, i] * sdev[i] * scale
    segments(
      center[1] - vec[1], center[2] - vec[2],
      center[1] + vec[1], center[2] + vec[2],
      col = cols[i], lwd = lwd, lty = i
    )
    if (show_labels) {
      text(
        center[1] + vec[1] * label_scale,
        center[2] + vec[2] * label_scale,
        labels = paste0("PC", i),
        col = cols[i], cex = cex,
        pos = if (vec[1] >= 0) 4 else 2,
        xpd = NA
      )
    }
  }
}

cov_from_eigensystem <- function(eigenvalues, rotation) {
  rotation %*% diag(eigenvalues, nrow = length(eigenvalues)) %*% t(rotation)
}

ellipsoid_surface_matrices <- function(center, sigma, level = 0.7,
                                       n_theta = 60, n_phi = 31) {
  ev <- eigen(sigma, symmetric = TRUE)
  theta <- seq(0, 2 * pi, length.out = n_theta)
  phi <- seq(0, pi, length.out = n_phi)
  x_sphere <- outer(sin(phi), cos(theta))
  y_sphere <- outer(sin(phi), sin(theta))
  z_sphere <- outer(cos(phi), rep(1, length(theta)))
  sphere <- rbind(c(x_sphere), c(y_sphere), c(z_sphere))
  radius <- sqrt(stats::qchisq(level, df = 3))
  shape <- ev$vectors %*% diag(sqrt(pmax(ev$values, 0)), nrow = 3)
  center <- as.numeric(center)
  coords <- center + radius * shape %*% sphere
  list(
    x = matrix(coords[1, ], nrow = length(phi), ncol = length(theta)),
    y = matrix(coords[2, ], nrow = length(phi), ncol = length(theta)),
    z = matrix(coords[3, ], nrow = length(phi), ncol = length(theta))
  )
}

regime_cols <- c("0" = "#3182bd", "1" = "#de2d26")


_This pkgdown-only HTML chunk was omitted from the Colab notebook._


## Introduction

This vignette introduces the Brownian-motion-based theory behind `bifrost`. The applied vignettes show full analyses on real datasets; the goal here is to develop the underlying ideas and the intuition behind them. [Whole-Tree Models, PCA, and bifrost](https://jakeberv.com/bifrost/articles/pca-model-selection-and-bifrost-vignette.html) takes up the model-selection side of the story.

At a high level, `bifrost` rests on three ideas:

1. trait evolution is modeled as a stochastic process on a phylogeny,
2. most empirical datasets are multivariate, so covariance matters as much as rate, and
3. branch-specific shifts are interesting because they let the multivariate rate structure change across the tree.

`bifrost` is built around Brownian-motion-based models, so most of what follows focuses on that family. Within that family, the current search is narrower than the full conceptual space described below: it targets branch-specific proportional scaling of a shared covariance geometry.


In [ ]:
library(bifrost)
library(ape)
library(phytools)
library(mvMORPH)


## 1. Why Stochastic Models?

Comparative datasets record the outcomes of many small processes acting over long stretches of time: mutation, drift, development, integration among traits, and changing environments. We rarely observe those microscopic steps directly. Instead, we model the distribution of possible outcomes.

That is the value of a stochastic model. It does **not** claim that evolution is unstructured. Instead, it treats uncertainty as something that can still be described with parameters. In comparative biology, those parameters usually control:

- an ancestral state or intercept,
- the rate at which variance accumulates through time, and
- the covariance structure among traits, together with the phylogenetically induced covariance among species.

The phylogeny is what makes this a comparative model rather than a generic random process. Branch lengths encode time, and shared ancestry tells us which species should be statistically similar before we even look at the trait matrix.

## 2. Brownian Motion on a Phylogeny

The simplest continuous-trait model is Brownian motion (BM). In one dimension,

$$
dX_t = \sigma dB_t,
$$

where \(B_t\) is standard Brownian motion and \(\sigma^2\) is the evolutionary rate. Under BM, the expected trait value stays centered on its starting point while variance grows linearly with time:

$$
\mathrm{Var}(X_t) = \sigma^2 t.
$$


In [ ]:
set.seed(0L)
times <- seq(0, 1, length.out = 250)

simulate_bm_path <- function(sigma2) {
  dt <- diff(times)
  c(0, cumsum(rnorm(length(dt), mean = 0, sd = sqrt(sigma2 * dt))))
}

n_paths_plot <- 20
slow_paths <- replicate(n_paths_plot, simulate_bm_path(0.10))
fast_paths <- replicate(n_paths_plot, simulate_bm_path(0.9))
n_batches_mean <- 250

draw_envelope <- function(x, lower, upper, fill) {
  polygon(
    x = c(x, rev(x)),
    y = c(lower, rev(upper)),
    col = fill,
    border = NA
  )
}

simulate_disparity_batches <- function(sigma2, n_batches, batch_size) {
  sapply(
    seq_len(n_batches),
    function(batch) {
      batch_paths <- replicate(batch_size, simulate_bm_path(sigma2))
      apply(batch_paths, 1, var)
    }
  )
}

slow_batches <- simulate_disparity_batches(0.10, n_batches_mean, n_paths_plot)
fast_batches <- simulate_disparity_batches(0.9, n_batches_mean, n_paths_plot)
slow_mean_disparity <- rowMeans(slow_batches)
fast_mean_disparity <- rowMeans(fast_batches)
slow_envelope_lower <- apply(slow_batches, 1, stats::quantile, probs = 0.025)
slow_envelope_upper <- apply(slow_batches, 1, stats::quantile, probs = 0.975)
fast_envelope_lower <- apply(fast_batches, 1, stats::quantile, probs = 0.025)
fast_envelope_upper <- apply(fast_batches, 1, stats::quantile, probs = 0.975)

ylim_paths <- range(slow_paths, fast_paths)
ylim_var <- range(
  slow_envelope_lower, slow_envelope_upper,
  fast_envelope_lower, fast_envelope_upper,
  slow_mean_disparity, fast_mean_disparity,
  0
)

op <- par(no.readonly = TRUE)
layout(matrix(c(1, 2), nrow = 1), widths = c(0.92, 1.08))
par(mgp = c(2.0, 0.65, 0))

par(mar = c(4.1, 4.1, 2.6, 1.0))
matplot(
  times, slow_paths,
  type = "n",
  xlab = "Time", ylab = "Trait value",
  xlim = range(times), ylim = ylim_paths
)
abline(h = 0, lty = 3, col = "grey65")
matlines(
  times, fast_paths,
  lty = 1,
  lwd = 1.1,
  col = rep("#f16913", n_paths_plot)
)
matlines(
  times, slow_paths,
  lty = 1,
  lwd = 1.45,
  col = rep("#31a354", n_paths_plot)
)
legend(
  "topleft",
  legend = c("Slow", "Fast"),
  col = c("#31a354", "#f16913"),
  lty = 1, lwd = c(1.45, 1.1), bty = "n", cex = 0.86
)
title(main = "Brownian trajectories", cex.main = 0.96, line = 0.8)

par(mar = c(4.1, 4.3, 2.6, 1.2))
plot(
  times, slow_mean_disparity,
  type = "n",
  xlab = "Time",
  ylab = "Disparity",
  xlim = range(times), ylim = ylim_var
)
draw_envelope(
  times,
  slow_envelope_lower,
  slow_envelope_upper,
  fill = grDevices::adjustcolor("#31a354", alpha.f = 0.10)
)
draw_envelope(
  times,
  fast_envelope_lower,
  fast_envelope_upper,
  fill = grDevices::adjustcolor("#f16913", alpha.f = 0.10)
)
lines(times, slow_mean_disparity, col = "#31a354", lwd = 2.2)
lines(times, fast_mean_disparity, col = "#f16913", lwd = 2.2)
legend(
  "topleft",
  legend = c("Slow", "Fast"),
  col = c("#31a354", "#f16913"),
  lty = 1,
  lwd = c(1.45, 1.1),
  cex = 0.86,
  bty = "n"
)
title(main = "Disparity through time", cex.main = 0.96, line = 0.8)

layout(1)
par(op)


_This pkgdown-only HTML chunk was omitted from the Colab notebook._


On a phylogeny, BM does more than generate trait variance through time: it also implies a covariance structure among species. Following [Felsenstein (1985)](https://doi.org/10.1086/284325), the covariance between species \(i\) and \(j\) is proportional to the amount of history they share from the root to their most recent common ancestor:

$$
\mathrm{Cov}(X_i, X_j) = \sigma^2 T_{ij}.
$$

That is the statistical reason close relatives tend to resemble each other under BM.


In [ ]:
small_tree <- ape::read.tree(text = "(((A:0.8,B:0.8):0.6,C:1.4):0.7,(D:0.6,E:0.6):1.5);")
shared_history <- ape::vcv.phylo(small_tree)
heat_cols <- colorRampPalette(c("#f7fbff", "#6baed6", "#08306b"))(80)
n_tips <- ape::Ntip(small_tree)

op <- par(no.readonly = TRUE)
layout(matrix(c(1, 2), nrow = 1), widths = c(0.95, 1.05))

par(mar = c(3.2, 0.8, 4.8, 1.2), mgp = c(1.9, 0.55, 0))
plot(small_tree, cex = 0.95, no.margin = FALSE, label.offset = 0.03)
title(main = "Tree topology and shared ancestry", line = 1.4, cex.main = 0.98)

tree_plot <- get("last_plot.phylo", envir = ape::.PlotPhyloEnv)
tip_y <- tree_plot$yy[seq_len(n_tips)]
tip_order <- small_tree$tip.label
shared_aligned <- shared_history[tip_order, tip_order]
shared_breaks <- seq(min(shared_aligned), max(shared_aligned), length.out = length(heat_cols) + 1)

par(mar = c(3.2, 4.0, 4.8, 1.0), mgp = c(1.9, 0.55, 0))
plot.new()
plot.window(xlim = c(0.5, n_tips + 0.5), ylim = range(tip_y) + c(-0.5, 0.5), asp = 1)

for (i in seq_len(n_tips)) {
  for (j in seq_len(n_tips)) {
    z <- shared_aligned[i, j]
    col_idx <- findInterval(z, shared_breaks, all.inside = TRUE)
    rect(
      xleft = j - 0.5, ybottom = tip_y[i] - 0.5,
      xright = j + 0.5, ytop = tip_y[i] + 0.5,
      col = heat_cols[col_idx], border = "white", lwd = 0.7
    )
  }
}

axis(1, at = seq_len(n_tips), labels = tip_order, las = 2, cex.axis = 0.85)
axis(2, at = tip_y, labels = tip_order, las = 2, cex.axis = 0.85)
box()
title(main = "Shared path length among tips", line = 1.4, cex.main = 0.98)

layout(1)
par(op)


The shared-history matrix for this toy tree is:


In [ ]:
# Display the toy tree's shared-history covariance matrix.
round(shared_history, 2)


BM is often described as a neutral or drift-like baseline, but that should be treated as intuition rather than a literal equivalence. A BM fit does **not** prove that selection was absent. It means the data are consistent with stochastic diffusion whose covariance is fully structured by the phylogeny.

## 3. From One Trait to Many

Most morphological and ecological datasets are not truly univariate. Landmarks, skeletal dimensions, climatic niches, and life-history syndromes all involve traits that can change together.

The natural multivariate extension of BM starts with trait-level covariance in \(\mathbf{\Sigma}\), then leads to proportional rate shifts, higher-dimensional eigenvalue structure, and finally the full phylogenetic covariance \( \mathbf{C} \otimes \mathbf{\Sigma} \).

At the simplest level, that extension replaces a single rate parameter with a variance-covariance matrix:

$$
\mathbf{\Sigma} =
\begin{bmatrix}
\sigma^2_1 & \sigma_{12} & \cdots \\
\sigma_{21} & \sigma^2_2 & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}.
$$

The diagonal entries are trait-specific rates, and the off-diagonal entries are evolutionary covariances. In other words, the diagonal tells us how quickly each trait changes, while the off-diagonal tells us whether traits tend to change together.

### Covariance geometry

One useful way to compare two matrices is by their eigenstructure. In the Flury hierarchy, the strongest similarity is equality; a weaker but still structured relationship is **proportionality**, where one matrix is just a scalar multiple of another; another is the **common principal components (CPC)** case, where matrices share their major axes but not necessarily the same eigenvalues; and a stronger difference changes both eigenvectors and eigenvalues ([Phillips and Arnold 1999](https://doi.org/10.1111/j.1558-5646.1999.tb05414.x)). As a rough visual guide, ellipse orientation reflects correlation structure, whereas ellipse size reflects overall rate ([Stepanova et al. 2025](https://doi.org/10.1093/sysbio/syaf002)).


In [ ]:
set.seed(3)
tree_mv <- phytools::pbtree(n = 60, scale = 1)

sigma_zero_cov <- matrix(c(0.65, 0.00,
                           0.00, 0.65), 2, 2, byrow = TRUE)
sigma_pos_cov <- matrix(c(0.65, 0.52,
                          0.52, 0.65), 2, 2, byrow = TRUE)

sim_zero_cov <- as_trait_matrix(mvMORPH::mvSIM(
  tree = tree_mv,
  nsim = 1,
  model = "BM1",
  param = list(ntraits = 2, sigma = sigma_zero_cov, theta = c(0, 0))
))
sim_pos_cov <- as_trait_matrix(mvMORPH::mvSIM(
  tree = tree_mv,
  nsim = 1,
  model = "BM1",
  param = list(ntraits = 2, sigma = sigma_pos_cov, theta = c(0, 0))
))

rownames(sim_zero_cov) <- tree_mv$tip.label
rownames(sim_pos_cov) <- tree_mv$tip.label

xlim_all <- range(c(sim_zero_cov[, 1], sim_pos_cov[, 1]))
ylim_all <- range(c(sim_zero_cov[, 2], sim_pos_cov[, 2]))

op <- par(mfrow = c(1, 2), mar = c(4.2, 4.2, 4.4, 1.3), mgp = c(1.9, 0.6, 0))

plot(
  sim_zero_cov[, 1], sim_zero_cov[, 2],
  pch = 19, col = "#31a354",
  xlab = "Trait 1", ylab = "Trait 2",
  main = "Zero covariance, equal rates",
  xlim = xlim_all, ylim = ylim_all, asp = 1
)
draw_cov_ellipse(colMeans(sim_zero_cov), cov(sim_zero_cov), border = "#31a354")

plot(
  sim_pos_cov[, 1], sim_pos_cov[, 2],
  pch = 19, col = "#756bb1",
  xlab = "Trait 1", ylab = "Trait 2",
  main = "Positive covariance, same rates",
  xlim = xlim_all, ylim = ylim_all, asp = 1
)
draw_cov_ellipse(colMeans(sim_pos_cov), cov(sim_pos_cov), border = "#756bb1")

par(op)


Figure 3 isolates covariance structure: both panels use the same diagonal rates, so the difference comes from the off-diagonal term rather than from faster or slower diffusion overall.

The matrices used in Figure 3 are:


In [ ]:
# Compare zero-covariance and positive-covariance trait matrices.
sigma_zero_cov
sigma_pos_cov
round(cov2cor(sigma_pos_cov), 2)


### Proportional rate shifts

A pure rate shift looks different. If \(\mathbf{\Sigma}_2 = c \mathbf{\Sigma}_1\) for some positive constant \(c\), the matrices are proportional in the Flury sense: they keep the same eigenvectors and the same relative eigenvalues, but one accumulates variance faster on every axis ([Phillips and Arnold 1999](https://doi.org/10.1111/j.1558-5646.1999.tb05414.x)).


In [ ]:
set.seed(13)

sigma_slow <- matrix(c(0.18, 0.11,
                       0.11, 0.14), 2, 2, byrow = TRUE)
sigma_fast <- 3 * sigma_slow

sim_slow <- as_trait_matrix(mvMORPH::mvSIM(
  tree = tree_mv,
  nsim = 1,
  model = "BM1",
  param = list(ntraits = 2, sigma = sigma_slow, theta = c(0, 0))
))
# Use the same realization so the plotted sample is exactly proportional.
sim_fast <- sqrt(3) * sim_slow

rownames(sim_slow) <- tree_mv$tip.label
rownames(sim_fast) <- tree_mv$tip.label

xlim_rate <- range(c(sim_slow[, 1], sim_fast[, 1]))
ylim_rate <- range(c(sim_slow[, 2], sim_fast[, 2]))

op <- par(mfrow = c(1, 2), mar = c(4.2, 4.2, 4.4, 1.3), mgp = c(1.9, 0.6, 0))

plot(
  sim_slow[, 1], sim_slow[, 2],
  pch = 19, col = "#2b8cbe",
  xlab = "Trait 1", ylab = "Trait 2",
  main = "Slower proportional regime",
  xlim = xlim_rate, ylim = ylim_rate, asp = 1
)
draw_cov_ellipse(colMeans(sim_slow), cov(sim_slow), border = "#2b8cbe")

plot(
  sim_fast[, 1], sim_fast[, 2],
  pch = 19, col = "#e6550d",
  xlab = "Trait 1", ylab = "Trait 2",
  main = "Faster proportional regime",
  xlim = xlim_rate, ylim = ylim_rate, asp = 1
)
draw_cov_ellipse(colMeans(sim_fast), cov(sim_fast), border = "#e6550d")

par(op)


That distinction also shows up in empirical work. Stepanova et al. (2025) used this Flury-inspired ladder explicitly when comparing scincid lizard clades, distinguishing single-matrix, proportional rate-shift, shared-eigenvector, and fully different-eigenvector models. The broader point here is that a multivariate regime shift need not mean the same thing in every case: sometimes it is mostly about tempo, and sometimes it reflects a deeper reorganization of trait covariation.

In Figure 4, \(\mathbf{\Sigma}_{\text{fast}} = 3 \mathbf{\Sigma}_{\text{slow}}\), so the faster regime is literally a proportional expansion of the slower one; the right panel is plotted as \(\sqrt{3}\) times the same simulated realization.

### Dimensionality and eigenvalues

It also helps to move from two traits to three. In three dimensions, covariance no longer just makes a cloud look round or elongated in a plane. It can make the cloud fill a volume, collapse toward a sheet, or narrow toward a ridge. In Figure 5, the three matrices all have the same trace, so the comparison is about how total variance is distributed across eigenvalues rather than about one regime simply having more variance overall.


_This pkgdown-only HTML chunk was omitted from the Colab notebook._


The 3D example makes the same point from another angle. With comparable total variance, evolution can still look effectively three-dimensional, nearly planar, or almost one-dimensional. A simple rule of thumb is that the number of comparatively large eigenvalues tells you how many dimensions the cloud really fills.

### From trait covariance to phylogenetic covariance

On a phylogeny, the full expected covariance has two ingredients: covariance among species and covariance among traits. A compact way to write it is \( \mathbf{C} \otimes \mathbf{\Sigma} \), where \(\mathbf{C}\) is the phylogenetic covariance matrix and \(\otimes\) is the Kronecker product.

One intuitive way to read that expression is one species pair at a time. For any pair of species \(i\) and \(j\), the corresponding trait-covariance block is just \(C_{ij}\mathbf{\Sigma}\). The main point is easy to lose in the notation: \(\mathbf{C}\) does not introduce a new trait-covariance pattern for every species pair. It simply rescales the same template according to how much history that pair shares. In Figure 6, \(C_{AB}\) stands in for one chosen entry from \(\mathbf{C}\). In the HTML version, the slider lets you change that scalar directly. As it moves, the A-B block grows or shrinks, but the pattern set by \(\mathbf{\Sigma}\) stays the same.


_This pkgdown-only HTML chunk was omitted from the Colab notebook._


The full matrix becomes easier to read if you think about it one species pair at a time. The same trait-covariance template is reused across every pair; what changes from block to block is only the scalar weight supplied by \(\mathbf{C}\). Put differently, \(\mathbf{C}\) controls how strong the covariance is between species, whereas \(\mathbf{\Sigma}\) controls the pattern of covariance across traits. That is the statistical backbone of multivariate phylogenetic GLS and of the `mvMORPH::mvgls` fits used inside `bifrost`.

## 4. What Does a Branch-Specific Shift Mean?
A single-regime BM model assumes that the same covariance structure applies everywhere on the tree. `bifrost` is built for cases where that assumption may be too simple, and the practical question becomes whether some clade appears to evolve under a different multivariate rate structure from the background.

As the examples above suggest, that difference can take several forms. In the Flury hierarchy, which compares covariance matrices by how much eigenstructure they share, one regime might differ mainly by proportional scaling, another might retain the same major axes while changing variance along them, and a more dramatic shift can alter both size and orientation ([Phillips and Arnold 1999](https://doi.org/10.1111/j.1558-5646.1999.tb05414.x)). Even when the fitted comparison is phrased as BM versus BMM rather than as an explicit Flury test, that vocabulary is still useful for interpretation.

A shift is not just a change in where traits sit. It is a change in the geometry of the evolving trait cloud. Here we show the more dramatic end of that spectrum, where a descendant regime changes both size and orientation. The current `bifrost` search is narrower: it tests the proportional-scaling case, where descendant regimes differ in overall rate while retaining the same covariance geometry.


In [ ]:
set.seed(4)
shift_tree <- phytools::pbtree(n = 60, scale = 1)
shift_node <- choose_shift_node(shift_tree, min_tips = 10, target_tips = 15)

baseline_tree <- phytools::paintBranches(
  shift_tree,
  edge = unique(shift_tree$edge[, 2]),
  state = "0",
  anc.state = "0"
)
shifted_tree <- phytools::paintSubTree(
  baseline_tree,
  node = shift_node,
  state = "1",
  anc.state = "0"
)

sigma_regimes <- list(
  "0" = matrix(c(0.12, 0.02,
                 0.02, 0.08), 2, 2, byrow = TRUE),
  "1" = matrix(c(0.90, 0.48,
                 0.48, 0.36), 2, 2, byrow = TRUE)
)

shift_sim <- as_trait_matrix(mvMORPH::mvSIM(
  tree = shifted_tree,
  nsim = 1,
  model = "BMM",
  param = list(ntraits = 2, sigma = sigma_regimes, theta = c(0, 0))
))
rownames(shift_sim) <- shifted_tree$tip.label
tip_regimes <- as.character(phytools::getStates(shifted_tree, type = "tips"))
shift_xlim <- grDevices::extendrange(shift_sim[, 1], f = 0.08)
shift_ylim <- grDevices::extendrange(shift_sim[, 2], f = 0.08)

op <- par(no.readonly = TRUE)
layout(matrix(c(1, 2), nrow = 1), widths = c(0.88, 1.12))
par(mar = c(3.2, 0.7, 4.6, 0.7), mgp = c(1.8, 0.55, 0))

phytools::plotSimmap(
  shifted_tree,
  colors = regime_cols,
  ftype = "off",
  lwd = 2,
  mar = c(3.0, 0.6, 4.9, 0.6),
  xlim = c(0, max(ape::node.depth.edgelength(shifted_tree)) * 1.12)
)
ape::nodelabels(node = shift_node, pch = 21, bg = "#ffd166", cex = 1.2)
title(main = "Accepted shift on the tree", line = 1.35, cex.main = 0.98)
legend(
  "bottomleft",
  legend = c("background regime", "shifted clade", "accepted shift"),
  col = c(regime_cols["0"], regime_cols["1"], NA),
  pch = c(15, 15, 21),
  pt.bg = c(NA, NA, "#ffd166"),
  bty = "o",
  box.col = NA,
  bg = grDevices::adjustcolor("white", alpha.f = 0.82),
  cex = 0.8,
  pt.cex = 1.05,
  y.intersp = 0.95,
  x.intersp = 0.75
)

par(mar = c(3.3, 3.9, 4.6, 1.0), mgp = c(1.9, 0.6, 0))
plot(
  shift_sim[, 1], shift_sim[, 2],
  col = regime_cols[tip_regimes], pch = 19,
  xlab = "Trait 1", ylab = "Trait 2",
  main = "Trait space under two regimes",
  asp = 1,
  cex = 0.92,
  xlim = shift_xlim,
  ylim = shift_ylim
)
for (regime in names(regime_cols)) {
  idx <- tip_regimes == regime
  draw_cov_ellipse(
    colMeans(shift_sim[idx, , drop = FALSE]),
    cov(shift_sim[idx, , drop = FALSE]),
    border = regime_cols[regime]
  )
}
legend(
  "bottomleft",
  legend = c("background tips", "shifted tips"),
  col = regime_cols,
  pch = 19,
  bty = "o",
  box.col = NA,
  bg = grDevices::adjustcolor("white", alpha.f = 0.82),
  cex = 0.8,
  pt.cex = 1.05,
  y.intersp = 0.95
)

layout(1)
par(op)


That example is conceptually useful, but it is not the literal target of `searchOptimalConfiguration()`. The search starts from a single-regime baseline, proposes internal nodes as candidate shifts, and asks whether allowing a descendant clade to occupy its own proportionally faster or slower multivariate regime gives a better approximation than the single-regime model, with enough improvement in fit to justify the added complexity.

For high-dimensional data, those comparisons are carried out with penalized-likelihood multivariate GLS machinery following the framework of [Clavel, Aristide, and Morlon (2019)](https://doi.org/10.1093/sysbio/syy045). In the returned `bifrost_search` object, `shift_nodes_no_uncertainty` stores the accepted shift locations, `VCVs` stores the regime-specific variance-covariance matrices under the fitted proportional-scaling model, and `baseline_ic`, `optimal_ic`, and `ic_weights` summarize support for those extra regimes.

## 5. A Quantitative-Genetic Bridge

Readers coming from quantitative genetics will recognize the same geometry in the additive genetic variance-covariance matrix, \(\mathbf{G}\). That resemblance is real. Both \(\mathbf{G}\) and the macroevolutionary rate matrix are covariance matrices, so the same language of eigenvalues, eigenvectors, major axes, proportionality, and lines of least resistance naturally carries across scales ([Lande 1979](https://doi.org/10.1111/j.1558-5646.1979.tb04694.x); [Schluter 1996](https://doi.org/10.1111/j.1558-5646.1996.tb03563.x); [Steppan, Phillips, and Houle 2002](https://www.bio.fsu.edu/dhoule/Publications/steppanetal2002.pdf)).

But the two matrices are not the same object. \(\mathbf{G}\) describes additive genetic covariance within populations. By contrast, \(\mathbf{\Sigma}\) or \(\mathbf{R}\) in a phylogenetic BM model describes the rate at which trait variance accumulates among lineages. Under a narrow drift-only null, with a stable \(\mathbf{G}\), constant effective population size, and branch lengths measured in generations, the expected evolutionary rate matrix is proportional to \(\mathbf{G}/N_e\) ([Lande 1979](https://doi.org/10.1111/j.1558-5646.1979.tb04694.x); [Hohenlohe and Arnold 2008](https://doi.org/10.1086/527498); [Revell and Harmon 2008](https://lukejharmon.github.io/assets/Revell_and_Harmon_2008.pdf)). Outside that special case, the macroevolutionary rate matrix can also reflect changing selection, moving optima, mutation, and evolution of \(\mathbf{G}\) itself, so it should not be read as a direct estimate of the within-population \(\mathbf{G}\)-matrix.


In [ ]:
set.seed(29)

rescale_trace <- function(sigma, target = 1) {
  sigma * (target / sum(diag(sigma)))
}

bridge_angle <- 34 * pi / 180
bridge_rotation <- matrix(
  c(cos(bridge_angle), -sin(bridge_angle),
    sin(bridge_angle),  cos(bridge_angle)),
  2, 2, byrow = TRUE
)
alt_angle <- -12 * pi / 180
alt_rotation <- matrix(
  c(cos(alt_angle), -sin(alt_angle),
    sin(alt_angle),  cos(alt_angle)),
  2, 2, byrow = TRUE
)

G_bridge <- cov_from_eigensystem(c(1.00, 0.22), bridge_rotation)
R_bridge <- 0.35 * G_bridge
R_alt_bridge <- cov_from_eigensystem(c(0.85, 0.10), alt_rotation)

G_shape <- rescale_trace(G_bridge, target = 1)
R_bridge_shape <- rescale_trace(R_bridge, target = 1)
R_alt_shape <- rescale_trace(R_alt_bridge, target = 1)

micro_pts <- scale(
  matrix(stats::rnorm(120 * 2), ncol = 2) %*% chol(G_shape),
  center = TRUE, scale = FALSE
)

bridge_tree <- phytools::pbtree(n = 200, scale = 1)
macro_drift <- scale(
  as_trait_matrix(mvMORPH::mvSIM(
    tree = bridge_tree,
    nsim = 1,
    model = "BM1",
    param = list(theta = c(0, 0), sigma = R_bridge_shape, ntraits = 2)
  )),
  center = TRUE, scale = FALSE
)
macro_alt <- scale(
  as_trait_matrix(mvMORPH::mvSIM(
    tree = bridge_tree,
    nsim = 1,
    model = "BM1",
    param = list(theta = c(0, 0), sigma = R_alt_shape, ntraits = 2)
  )),
  center = TRUE, scale = FALSE
)

bridge_x <- c(micro_pts[, 1], macro_drift[, 1], macro_alt[, 1])
bridge_y <- c(micro_pts[, 2], macro_drift[, 2], macro_alt[, 2])
bridge_xrange <- grDevices::extendrange(bridge_x, f = 0.06)
bridge_yrange <- grDevices::extendrange(bridge_y, f = 0.06)
bridge_span <- max(diff(bridge_xrange), diff(bridge_yrange))
bridge_center <- c(mean(bridge_xrange), mean(bridge_yrange))
bridge_xlim <- bridge_center[1] + c(-0.5, 0.5) * bridge_span
bridge_ylim <- bridge_center[2] + c(-0.5, 0.5) * bridge_span

op <- par(no.readonly = TRUE)
layout(matrix(1:3, nrow = 1), widths = c(1.10, 1, 1))
par(mgp = c(2.75, 0.98, 0), cex.lab = 2.58, cex.axis = 2.34, xaxs = "i", yaxs = "i", pty = "s")

panel_box <- function(title) {
  title_parts <- strsplit(title, "\n", fixed = TRUE)[[1]]
  mtext(title_parts[1], side = 3, line = 3.2, cex = 2.0, font = 2)
  if (length(title_parts) > 1) {
    mtext(title_parts[2], side = 3, line = 1.3, cex = 1.5, font = 1, col = "#7b8190")
  }
}

par(mar = c(4.15, 5.20, 6.35, 0.42))
plot(
  micro_pts[, 1], micro_pts[, 2],
  col = grDevices::adjustcolor("#31a354", alpha.f = 0.35),
  pch = 19, cex = 1.0,
  xlab = "Trait 1", ylab = "Trait 2",
  xlim = bridge_xlim, ylim = bridge_ylim,
  asp = 1
)
draw_cov_ellipse(c(0, 0), G_shape, border = "#238b45", lwd = 2.55)
points(0, 0, pch = 21, bg = "#238b45", col = "white", cex = 1.05)
panel_box("Within-population G\nadditive genetic covariance")
legend(
  "bottomright",
  legend = c("individuals", "G ellipse"),
  col = c(grDevices::adjustcolor("#31a354", alpha.f = 0.75), "#238b45"),
  pch = c(19, NA),
  pt.cex = c(1.2, NA),
  lty = c(0, 1),
  lwd = c(0, 2.55),
  bty = "o",
  box.col = NA,
  bg = grDevices::adjustcolor("white", alpha.f = 0.88),
  inset = 0.01,
  cex = 1.55,
  y.intersp = 1.05,
  seg.len = 2.9
)

par(mar = c(4.15, 2.15, 6.35, 0.55))
plot(
  macro_drift[, 1], macro_drift[, 2],
  col = grDevices::adjustcolor("#3182bd", alpha.f = 0.65),
  pch = 19, cex = 1.2,
  xlab = "Trait 1", ylab = "",
  xlim = bridge_xlim, ylim = bridge_ylim,
  yaxt = "n",
  asp = 1
)
draw_cov_ellipse(c(0, 0), G_shape, border = "#9aa5b1", lwd = 8.09, lty = 2, lend = "butt")
draw_cov_ellipse(c(0, 0), R_bridge_shape, border = "#2171b5", lwd = 2.8)
points(0, 0, pch = 21, bg = "#2171b5", col = "white", cex = 1.05)
panel_box("Drift-only null\nR proportional to G / Ne")
old_lend <- par("lend")
par(lend = "butt")
legend(
  "bottomright",
  legend = c("standardized G", "standardized R"),
  col = c("#9aa5b1", "#2171b5"),
  lty = c(2, 1),
  lwd = c(6.47, 2.55),
  bty = "o",
  box.col = NA,
  bg = grDevices::adjustcolor("white", alpha.f = 0.88),
  inset = 0.01,
  cex = 1.72,
  y.intersp = 1.08,
  seg.len = 5.4
)
par(lend = old_lend)

par(mar = c(4.15, 2.15, 6.35, 0.55))
plot(
  macro_alt[, 1], macro_alt[, 2],
  col = grDevices::adjustcolor("#e6550d", alpha.f = 0.68),
  pch = 19, cex = 1.2,
  xlab = "Trait 1", ylab = "",
  xlim = bridge_xlim, ylim = bridge_ylim,
  yaxt = "n",
  asp = 1
)
draw_cov_ellipse(c(0, 0), G_shape, border = "#9aa5b1", lwd = 2.15, lty = 2)
draw_cov_ellipse(c(0, 0), R_alt_shape, border = "#d94801", lwd = 2.55)
points(0, 0, pch = 21, bg = "#d94801", col = "white", cex = 1.05)
panel_box("Relax the null\nselection or evolving G")
legend(
  "bottomright",
  legend = c("standardized G", "standardized R"),
  col = c("#9aa5b1", "#d94801"),
  lty = c(2, 1),
  lwd = c(2.15, 2.55),
  bty = "o",
  box.col = NA,
  bg = grDevices::adjustcolor("white", alpha.f = 0.88),
  inset = 0.01,
  cex = 1.6,
  y.intersp = 1.06,
  seg.len = 2.9
)

layout(1)
par(op)


## Practical Takeaways

- Brownian motion is useful because, on a phylogeny, it turns shared evolutionary history into an expected covariance structure among species.
- In multivariate data, the covariance matrix is part of the biological signal, not just a nuisance parameter.
- Regime differences can reflect overall rate scaling, changes in covariance geometry, or both.
- A branch-specific shift means that some clade is better described by a different multivariate rate structure than the background; in current `bifrost`, the search targets the proportional-scaling case.
- Under a narrow drift-only null, macroevolutionary rate geometry can be proportional to \(\mathbf{G}/N_e\), but the phylogenetic rate matrix should not be read as the within-population \(\mathbf{G}\)-matrix itself.
- High-dimensional datasets require regularization and conservative model comparison, which is why `bifrost` relies on penalized-likelihood `mvgls` machinery.

## Next Steps

If you want to keep going from here:

- Part 2, [Whole-Tree Models, PCA, and bifrost](https://jakeberv.com/bifrost/articles/pca-model-selection-and-bifrost-vignette.html), takes up whole-tree model limits, PCA-related cautions, and model-selection artifacts;
- the [jaw-shape vignette](https://jakeberv.com/bifrost/articles/jaw-shape-vignette.html) shows an intercept-only morphometric workflow on Paleozoic fishes.

## References

- Clavel, J., Aristide, L., and Morlon, H. (2019). *A Penalized Likelihood Framework for High-Dimensional Phylogenetic Comparative Methods and an Application to New-World Monkeys Brain Evolution*. [https://doi.org/10.1093/sysbio/syy045](https://doi.org/10.1093/sysbio/syy045)
- Felsenstein, J. (1985). *Phylogenies and the Comparative Method*. [https://doi.org/10.1086/284325](https://doi.org/10.1086/284325)
- Hohenlohe, P. A., and Arnold, S. J. (2008). *MIPoD: A Hypothesis-Test Framework for Microevolutionary Inference From Patterns of Divergence*. [https://doi.org/10.1086/527498](https://doi.org/10.1086/527498)
- Lande, R. (1979). *Quantitative Genetic Analysis of Multivariate Evolution, Applied to Brain:Body Size Allometry*. [https://doi.org/10.1111/j.1558-5646.1979.tb04694.x](https://doi.org/10.1111/j.1558-5646.1979.tb04694.x)
- Phillips, P. C., and Arnold, S. J. (1999). *Hierarchical Comparison of Genetic Variance-Covariance Matrices. I. Using the Flury Hierarchy*. [https://doi.org/10.1111/j.1558-5646.1999.tb05414.x](https://doi.org/10.1111/j.1558-5646.1999.tb05414.x)
- Revell, L. J., and Harmon, L. J. (2008). *Testing Quantitative Genetic Hypotheses About the Evolutionary Rate Matrix for Continuous Characters*. [https://lukejharmon.github.io/assets/Revell_and_Harmon_2008.pdf](https://lukejharmon.github.io/assets/Revell_and_Harmon_2008.pdf)
- Schluter, D. (1996). *Adaptive Radiation Along Genetic Lines of Least Resistance*. [https://doi.org/10.1111/j.1558-5646.1996.tb03563.x](https://doi.org/10.1111/j.1558-5646.1996.tb03563.x)
- Steppan, S. J., Phillips, P. C., and Houle, D. (2002). *Comparative Quantitative Genetics: Evolution of the G Matrix*. [https://www.bio.fsu.edu/dhoule/Publications/steppanetal2002.pdf](https://www.bio.fsu.edu/dhoule/Publications/steppanetal2002.pdf)
- Stepanova, N., Boyko, J. D., Lin, J., Davis Rabosky, A. R., and Rabosky, D. L. (2025). *Punctuated Versus Gradual Shifts in the Multivariate Evolutionary Process: A Test With Paired Radiations of Scincid Lizards*. [https://doi.org/10.1093/sysbio/syaf002](https://doi.org/10.1093/sysbio/syaf002)

## Software Used in This Vignette

- Clavel, J., Escarguel, G., and Merceron, G. (2015). *mvMORPH: an R package for fitting multivariate evolutionary models to morphometric data*. *Methods in Ecology and Evolution*, 6(11), 1311-1319. [https://doi.org/10.1111/2041-210X.12420](https://doi.org/10.1111/2041-210X.12420)
- Ligges, U., and Mächler, M. (2003). *Scatterplot3d - an R Package for Visualizing Multivariate Data*. *Journal of Statistical Software*, 8(11), 1-20. [https://doi.org/10.18637/jss.v008.i11](https://doi.org/10.18637/jss.v008.i11)
- Paradis, E., and Schliep, K. (2019). *ape 5.0: an environment for modern phylogenetics and evolutionary analyses in R*. *Bioinformatics*, 35, 526-528. [https://doi.org/10.1093/bioinformatics/bty633](https://doi.org/10.1093/bioinformatics/bty633)
- Revell, L. J. (2024). *phytools 2.0: an updated R ecosystem for phylogenetic comparative methods (and other things).* *PeerJ*, 12, e16505. [https://doi.org/10.7717/peerj.16505](https://doi.org/10.7717/peerj.16505)
- Sievert, C. (2020). *Interactive Web-Based Data Visualization with R, plotly, and shiny*. Chapman and Hall/CRC. [https://plotly-r.com](https://plotly-r.com)
- Cheng, J., Sievert, C., Schloerke, B., Chang, W., Xie, Y., and Allen, J. (2024). *htmltools: Tools for HTML*. R package version 0.5.8.1. [https://CRAN.R-project.org/package=htmltools](https://CRAN.R-project.org/package=htmltools)

## AI Assistance

This vignette was developed with assistance from OpenAI tools for drafting, editing, and figure refinement; all scientific content, interpretation, and final decisions were reviewed by the authors.
